# Qwen2.5-14B Quantization Logit Payload Generator

This notebook generates float64 next-token logit payloads for the Qwen2.5-14B quantization follow-up experiment.

Goal: fix the 14B reference model family and compare different quantization/loading variants using the same six prompts from the primary prompt inventory.

Use this notebook once per quantization/loading setting. Keep `MODEL_KEY` fixed at `14b`, update `LOAD_MODE`, then save six payloads with filenames that encode the loading setting.


## Step 1: Install Dependencies

This installs the Hugging Face loading stack. `bitsandbytes` is needed only if you choose `8bit` or `4bit` loading.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece safetensors

## Step 2: Check Your GPU

This does not change anything. It just shows which GPU Colab gave you.

In [ ]:
!nvidia-smi

## Step 3: Choose and Safely Load One Model

Edit the model/loading variables in the next code cell, then run it. The loader automatically cleans up any previously loaded model before loading the new one. To switch models later, change the variables and rerun only the model-loading cell.


In [ ]:
# Optional storage/memory cleanup before loading a model.
# This cell is self-contained so the notebook can run cleanly from a fresh runtime.
import gc

for variable_name in ("model", "tokenizer"):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as exc:
    print("Torch cleanup skipped:", repr(exc))

# Clear Hugging Face and Torch caches from the default Colab locations.
# Use this when you intentionally want to free storage before a different model run.
!rm -rf /root/.cache/huggingface
!rm -rf /root/.cache/torch
!rm -rf /content/.cache/huggingface
!rm -rf /content/.cache/torch

# Clear pip/cache/temp junk.
!rm -rf /root/.cache/pip
!rm -rf /tmp/*

# Optional: remove local model output/logit files only after downloading them.
# !rm -f /content/*.pt
# !rm -f /content/*.zip

# Show remaining disk use.
!du -h --max-depth=2 /root 2>/dev/null | sort -hr | head -20
!du -h --max-depth=2 /content 2>/dev/null | sort -hr | head -20
!df -h


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_KEY = "14b"      # Fixed for this experiment. Vary LOAD_MODE, not MODEL_KEY.
LOAD_MODE = "8bit"    # "bf16", "fp16", "8bit", or "4bit"

MODELS = {
    "14b": "Qwen/Qwen2.5-14B-Instruct",
}


def cleanup_loaded_model():
    """Release the currently loaded model/tokenizer, if they exist, before switching models."""
    import gc
    global model, tokenizer

    if "model" in globals():
        del model
    if "tokenizer" in globals():
        del tokenizer

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_selected_model(model_key, load_mode):
    """
    Safely load the fixed Qwen2.5-14B model and return (model, tokenizer, model_id).

    This mirrors the primary experiment prompt-selection notebook. For the
    quantization experiment, MODEL_KEY is intentionally fixed at '14b'; change
    LOAD_MODE to compare quantization/loading settings.
    """
    cleanup_loaded_model()

    if model_key != "14b":
        raise ValueError("The quantization experiment fixes MODEL_KEY='14b'; vary LOAD_MODE instead.")

    selected_model_id = MODELS[model_key]
    quantization_config = None
    torch_dtype = None

    if load_mode == "bf16":
        torch_dtype = torch.bfloat16
    elif load_mode == "fp16":
        torch_dtype = torch.float16
    elif load_mode == "8bit":
        quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    elif load_mode == "4bit":
        quantization_config = BitsAndBytesConfig(load_in_4bit=True)
    else:
        raise ValueError("LOAD_MODE must be 'bf16', 'fp16', '8bit', or '4bit'.")

    selected_tokenizer = AutoTokenizer.from_pretrained(selected_model_id)
    selected_model = AutoModelForCausalLM.from_pretrained(
        selected_model_id,
        device_map="auto",
        torch_dtype=torch_dtype,
        quantization_config=quantization_config,
    )

    selected_model.eval()
    print("Fixed model:", selected_model_id)
    print("Selected quantization/load mode:", load_mode)
    print(f"Initialized {selected_model_id} using LOAD_MODE={load_mode}")
    return selected_model, selected_tokenizer, selected_model_id


model, tokenizer, model_id = load_selected_model(MODEL_KEY, LOAD_MODE)


## Step 4: Model Switching Reminder

To switch models, go back to Step 3, change the model/loading variables, and rerun only that cell. The old model is cleaned up automatically before the new one loads.


In [ ]:
# No action needed here. Model switching is handled in the model-loading cell above.
# Current model:
print(model_id)


## Step 5: Define Float64 Logit Extraction and Save Helpers

The main helper is `save_next_token_logit_payload(...)`. Give it a context and filename, and it saves everything needed for later analysis: CPU float64 next-token logits, the plain context, the chat-formatted context, the formatted token ID sequence, and lightweight model metadata.

Precision convention:

- `payload["logits"]` is saved as a CPU float64 tensor.
- Float64 is a lossless widening conversion for the fp16, bf16, and fp32 logit values normally emitted by Hugging Face LLM inference.
- This does not recover precision that was never present inside quantized/bf16/fp16 inference; it simply prevents the notebook from introducing additional precision loss after extraction.


In [ ]:
from pathlib import Path


PAYLOAD_SCHEMA_VERSION = "next_token_logits_float64_v1"


def _logits_to_float64_cpu(raw_logits):
    """Return a CPU float64 copy of a one-dimensional logit vector.

    Float64 is a lossless widening conversion for the fp16, bf16, and fp32 logit
    dtypes normally produced by Hugging Face LLM inference. It does not recover
    precision that was never present in the model output, but it prevents the
    notebook from introducing additional precision loss when saving or analyzing
    logits.
    """
    if raw_logits.ndim != 1:
        raise ValueError("Expected a one-dimensional next-token logit vector")
    return raw_logits.detach().cpu().to(dtype=torch.float64).contiguous()


def format_context_for_qwen_chat(tokenizer, context):
    """
    Convert your plain context string into the exact chat-formatted string Qwen sees.

    Parameters
    ----------
    tokenizer:
        The Hugging Face tokenizer loaded for the selected Qwen model.
        It knows Qwen's required chat template.
    context:
        The plain text prompt/context you want to test. This should be the user-side
        context, before Qwen-specific chat formatting is added.

    Returns
    -------
    str
        The raw formatted text that will be tokenized and fed into the model.
    """
    messages = [{"role": "user", "content": context}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def get_next_token_logits(model, tokenizer, context):
    """
    Run the model once and return the next-token logit vector as CPU float64.

    The returned tensor has one logit per vocabulary token. The float64 conversion
    is a lossless widening conversion for fp16/bf16/fp32 model-output logits and
    is the safest dtype for downstream probability calculations.
    """
    formatted_context = format_context_for_qwen_chat(tokenizer, context)
    inputs = tokenizer(formatted_context, return_tensors="pt").to(model.device)

    model.eval()
    with torch.inference_mode():
        outputs = model(**inputs)

    return _logits_to_float64_cpu(outputs.logits[0, -1, :])


def build_next_token_logit_payload(model, tokenizer, context, *, model_id=None, load_mode=None):
    """
    Build the complete saved object for one context without writing it to disk.

    The payload saves `payload["logits"]` as a CPU float64 tensor. Float64 avoids
    additional precision loss after extraction while keeping the payload simple.
    """
    formatted_context = format_context_for_qwen_chat(tokenizer, context)
    inputs = tokenizer(formatted_context, return_tensors="pt").to(model.device)

    model.eval()
    with torch.inference_mode():
        outputs = model(**inputs)

    logits = _logits_to_float64_cpu(outputs.logits[0, -1, :])
    input_ids = inputs.input_ids[0].detach().cpu().long().contiguous()

    return {
        "payload_schema_version": PAYLOAD_SCHEMA_VERSION,
        "logits": logits,
        "context": context,
        "formatted_context": formatted_context,
        "formatted_context_token_ids": input_ids,
        "model_id": model_id,
        "load_mode": load_mode,
        "logits_dtype": str(logits.dtype),
        "logits_shape": tuple(logits.shape),
        "vocab_size": int(logits.shape[0]),
        "formatted_context_length_tokens": int(input_ids.shape[0]),
    }


def save_logit_vector(logits, filename, *, context=None, formatted_context=None, formatted_context_token_ids=None, model_id=None, load_mode=None):
    """
    Save an externally supplied logit vector plus useful notes into a .pt file.

    Prefer save_next_token_logit_payload(...) when possible. This helper exists for
    manual vectors and compatibility.
    """
    path = Path(filename)
    if path.suffix == "":
        path = path.with_suffix(".pt")

    path.parent.mkdir(parents=True, exist_ok=True)
    logits64 = _logits_to_float64_cpu(logits)
    token_ids = formatted_context_token_ids.detach().cpu().long().contiguous() if formatted_context_token_ids is not None else None

    payload = {
        "payload_schema_version": PAYLOAD_SCHEMA_VERSION,
        "logits": logits64,
        "context": context,
        "formatted_context": formatted_context,
        "formatted_context_token_ids": token_ids,
        "model_id": model_id,
        "load_mode": load_mode,
        "logits_dtype": str(logits64.dtype),
        "logits_shape": tuple(logits64.shape),
        "vocab_size": int(logits64.shape[0]),
        "formatted_context_length_tokens": int(token_ids.shape[0]) if token_ids is not None else None,
    }
    torch.save(payload, path)
    print(f"Saved float64 logits to {path}")
    return path


def save_next_token_logit_payload(model, tokenizer, context, filename, *, model_id=None, load_mode=None):
    """
    Main user-facing helper: save everything needed for one context in one call.

    The saved .pt file contains CPU float64 next-token logits, the original
    context, the exact chat-formatted context, formatted token IDs, and lightweight
    model metadata.
    """
    path = Path(filename)
    if path.suffix == "":
        path = path.with_suffix(".pt")

    path.parent.mkdir(parents=True, exist_ok=True)
    payload = build_next_token_logit_payload(
        model,
        tokenizer,
        context,
        model_id=model_id,
        load_mode=load_mode,
    )
    payload["saved_path"] = str(path)
    torch.save(payload, path)
    print(f"Saved float64 logits and context tokens to {path}")
    return payload


def get_and_save_next_token_logits(model, tokenizer, context, filename):
    """
    Convenience helper: get logits for one context and immediately save them.

    Returns
    -------
    torch.Tensor
        The saved CPU float64 logit vector.
    """
    payload = save_next_token_logit_payload(
        model,
        tokenizer,
        context,
        filename,
        model_id=model_id,
        load_mode=LOAD_MODE,
    )
    return payload["logits"]


def save_many_next_token_logit_payloads(model, tokenizer, context_filename_pairs, *, model_id=None, load_mode=None):
    """
    Main batch helper: save complete payloads for several contexts in one loop.
    """
    saved = {}
    for context, filename in context_filename_pairs:
        payload = save_next_token_logit_payload(
            model,
            tokenizer,
            context,
            filename,
            model_id=model_id,
            load_mode=load_mode,
        )
        saved[filename] = payload
    return saved


def top_k_logit_tokens(logits, tokenizer, k=10):
    """
    Return the k highest-logit tokens as IDs, strings, logits, and base probabilities.

    Probabilities are computed in float64 with a stable softmax.
    """
    if logits.ndim != 1:
        raise ValueError("logits must be a one-dimensional tensor")
    if k <= 0:
        raise ValueError("k must be positive")

    k = min(int(k), int(logits.shape[0]))
    logits_cpu = logits.detach().cpu().to(dtype=torch.float64)
    shifted_logits = logits_cpu - torch.max(logits_cpu)
    probs_cpu = torch.exp(shifted_logits)
    probs_cpu = probs_cpu / torch.sum(probs_cpu)
    values, token_ids = torch.topk(logits_cpu, k=k)

    rows = []
    for token_id, value in zip(token_ids.tolist(), values.tolist()):
        rows.append({
            "token_id": token_id,
            "token_string": tokenizer.decode([token_id]),
            "logit": float(value),
            "base_probability": float(probs_cpu[token_id]),
        })
    return rows


def print_top_k_logit_tokens(logits, tokenizer, k=10):
    """Print top_k_logit_tokens(...) in a compact notebook-friendly table."""
    rows = top_k_logit_tokens(logits, tokenizer, k=k)
    for rank, row in enumerate(rows, start=1):
        token_text = repr(row["token_string"])
        print(f"{rank:>2}. id={row['token_id']:<8} logit={row['logit']:>12.8g} prob={row['base_probability']:.12g} token={token_text}")
    return rows


## Step 6: Sandbox: Inspect Formatted Input and Generate a Response

Use this section to test contexts before saving logits. Paste any sandbox context, inspect the exact raw string after Qwen chat formatting, and generate a normal model response from that same formatted input. Repeat this step as much as needed before moving on.


In [ ]:
SANDBOX_CONTEXT = "Output a random binary sequence and nothing else. I should not be seeing words in your response."

sandbox_formatted_context = format_context_for_qwen_chat(tokenizer, SANDBOX_CONTEXT)
print(sandbox_formatted_context)

In [ ]:
sandbox_inputs = tokenizer(sandbox_formatted_context, return_tensors="pt").to(model.device)

with torch.no_grad():
    sandbox_output_ids = model.generate(
        **sandbox_inputs,
        max_new_tokens=1,
        do_sample=True,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

sandbox_new_tokens = sandbox_output_ids[0, sandbox_inputs.input_ids.shape[-1]:]
print(tokenizer.decode(sandbox_new_tokens, skip_special_tokens=True))
print("token id:", sandbox_new_tokens.item())

## Step 7: Generate and Save Six 14B Quantization Payloads

After the sandbox looks right, run the six prompt inventory for the current 14B quantization/loading setting. These prompts are intentionally the same six prompts from the primary prompt inventory.


In [ ]:
FILE_PREFIX = f"qwen25_{MODEL_KEY}_{LOAD_MODE}"

CONTEXTS_TO_SAVE = [
    (
        "Output a random binary sequence and nothing else. I should not be seeing words in your response.",
        f"{FILE_PREFIX}_prompt_1_logits.pt",
    ),
    (
        "Generate a RANDOM number from {zero, one}. Your response should be a single numerical digit. Nothing else. Try to be as unpredictable as possible.",
        f"{FILE_PREFIX}_prompt_2_logits.pt",
    ),
    (
        "Generate a highly random sequence of zero's and one's. Your response should be a string of numerical digits and nothing else. Do NOT output non-numerical text.",
        f"{FILE_PREFIX}_prompt_3_logits.pt",
    ),
    (
        "Simulate a completely random coin flip. Output the number zero if it's heads. Output the number one if it's tails. Your output should be a single numerical digit. DON'T GIVE THE SAME RESPONSE EVERY TIME. TRY TO MAKE IT A FIFTY-FIFTY CHANCE.",
        f"{FILE_PREFIX}_prompt_4_logits.pt",
    ),
    (
        "Generate a completely random comma-separated list of zero's and ones. Each entry in the list must be a numerical digit. Do not output anything else except this list.",
        f"{FILE_PREFIX}_prompt_5_logits.pt",
    ),
    (
        "OUTPUT THE NUMBER ZERO OR THE NUMBER ONE. MAKE IT A FIFTY-FIFTY CHANCE. YOUR RESPONSE SHOULD BE A NUMERICAL DIGIT AND NOTHING ELSE.",
        f"{FILE_PREFIX}_prompt_6_logits.pt",
    ),
]

saved_payloads = save_many_next_token_logit_payloads(
     model,
     tokenizer,
     CONTEXTS_TO_SAVE,
     model_id=model_id,
     load_mode=LOAD_MODE,
 )

## Step 8: Load and Inspect the Six Saved Logit Payloads

Use this after saving the six prompt payloads. It loads each file, verifies the saved metadata, prints the base probabilities for token `0` and token `1`, and prints the top-k highest-logit tokens by token ID and decoded string.


In [ ]:
TOP_K = 10
ZERO_TOKEN_ID = 15
ONE_TOKEN_ID = 16
SAVED_FILES = [filename for _, filename in CONTEXTS_TO_SAVE]


def stable_softmax_float64(logits):
    logits64 = logits.detach().cpu().to(dtype=torch.float64)
    shifted = logits64 - torch.max(logits64)
    probs = torch.exp(shifted)
    return probs / torch.sum(probs)


loaded_payloads = {}
probability_summary = []

for prompt_index, saved_file in enumerate(SAVED_FILES, start=1):
    print("=" * 80)
    print("prompt:", prompt_index)
    print("file:", saved_file)

    payload = torch.load(saved_file, map_location="cpu")
    loaded_payloads[saved_file] = payload

    loaded_logits = payload["logits"]
    loaded_context_token_ids = payload["formatted_context_token_ids"]
    loaded_probs = stable_softmax_float64(loaded_logits)
    p_zero = float(loaded_probs[ZERO_TOKEN_ID])
    p_one = float(loaded_probs[ONE_TOKEN_ID])
    probability_summary.append({
        "prompt": prompt_index,
        "file": saved_file,
        "p_zero": p_zero,
        "p_one": p_one,
        "p_zero_plus_p_one": p_zero + p_one,
    })

    print("model_id:", payload.get("model_id"))
    print("load_mode:", payload.get("load_mode"))
    print("payload_schema_version:", payload.get("payload_schema_version"))
    print("logit shape:", tuple(loaded_logits.shape))
    print("logit dtype:", loaded_logits.dtype)
    print("logit device:", loaded_logits.device)
    print("formatted context token count:", len(loaded_context_token_ids))
    print("first 20 formatted context token ids:", loaded_context_token_ids[:20].tolist())
    print("P(token '0', id 15):", f"{p_zero:.12g}")
    print("P(token '1', id 16):", f"{p_one:.12g}")
    print("P('0') + P('1'):", f"{p_zero + p_one:.12g}")
    print(f"top {TOP_K} tokens by raw logit:")
    print_top_k_logit_tokens(loaded_logits, tokenizer, k=TOP_K)
    print()

print("=" * 80)
print("Probability summary for target tokens")
for row in probability_summary:
    print(
        f"prompt {row['prompt']}: "
        f"P(0)={row['p_zero']:.12g}, "
        f"P(1)={row['p_one']:.12g}, "
        f"P(0)+P(1)={row['p_zero_plus_p_one']:.12g}, "
        f"file={row['file']}"
    )
